In [3]:
Interesting_regions =   [4, 5, 32, 33, 44, 45, 48, 49]

Good_subjects =         [0, 2, 5, 9, 12, 14]
Ok_subjects =           [1, 3, 4, 6, 11, 13, 19]
Other_subjects = [i for i in range(20) if i not in Ok_subjects and i not in Good_subjects]

Subjects_dict = {"Good_subjects": Good_subjects, "Ok_subjects":Ok_subjects, "Other_subjects":Other_subjects}

data_folder="/Users/giovanni.messuti/Desktop/BCI_Project/Data/"

freq_personalized = {"subject_0":[9,14.5],
                     "subject_1":[8,12],
                     "subject_2":[8,13.5],
                     "subject_3":[4,9.5],
                     "subject_4":[18,23.5],
                     "subject_5":[4,9.5],
                     "subject_6":[4,9.5],
                     "subject_7":[4,9.5],
                     "subject_8":[4,12],
                     "subject_9":[9.5,15.5],
                     "subject_10":[9,14.5],
                     "subject_11":[4,10.5],
                     "subject_12":[8,13.5],
                     "subject_13":[4,10.],
                     "subject_14":[12,18],
                     "subject_15":[4,8],
                     "subject_16":[4,13],
                     "subject_17":[4,12],
                     "subject_18":[4,9.5],
                     "subject_19":[4,12.5],
}

from BCI_Library import read_subject, plot_brain, train_models, extract_features_more_bands, saveresults_pickle, select_rows, compute_welch_select_regions, extract_features_from_spectra_more_bands

"""# new_row = {
#     "ROIs": [selected_regions_EEG],
#     "NROIs":[25]
#     "DataType": ["EEG"],
#     "freqs_band": [((4,8),(8,12),(12,30))],  
#     "subject": [subject_index],
#     "Classifier": ["LDA"],
#     "Performance": [training[0]],
#     "Features": ["PowerSpectra - MeanMax band"],
#     "Comments": ["-"],
#     "Date": [today],
#     "Personalized_Freq_Range" = False,
# }"""


'# new_row = {\n#     "ROIs": [selected_regions_EEG],\n#     "NROIs":[25]\n#     "DataType": ["EEG"],\n#     "freqs_band": [((4,8),(8,12),(12,30))],  \n#     "subject": [subject_index],\n#     "Classifier": ["LDA"],\n#     "Performance": [training[0]],\n#     "Features": ["PowerSpectra - MeanMax band"],\n#     "Comments": ["-"],\n#     "Date": [today],\n#     "Personalized_Freq_Range" = False,\n# }'

# Parallalize Trainings

In [ ]:
import mne
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from datetime import date
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA

today = date.today()
sfreq = 250


Rows = pd.DataFrame({
    "ROIs": [],
    "NROIs":[],
    "DataType": [],
    "freqs_band": [],  
    "subject": [],
    "Classifier": [],
    "Performance": [],
    "Features": [],
    "Comments": [],
    "Date": []
})

# keys of the performance dictionary
saveresults = True
filepath = "/Users/giovanni.messuti/Desktop/BCI_Project/Results/BCI_Performances.pkl"
DataType="EEG"
miff = [4,8,12]
maff = [8,12,30]
freqs_band = tuple([(i,j) for i,j in zip(miff,maff)])
Classifier_name = "RandomForest" #SVC, #RandomForest, #LDA
meth = RandomForestClassifier
Features_name = "PowerSpectra - MeanMax band"
Comments= "-" # "-"
PFR = False


for subject_index in range(20):
    for num_best_ROIs in range(14,69,3):
        print(f"\n\n Subject {subject_index}")
        data_2_MI = read_subject(data_folder, DataType, "MI",subject_index, verbose=False)
        data_2_Rest =read_subject(data_folder, DataType, "Baseline",subject_index, verbose=False)
        #####################################################################################################################
        fmin=3
        fmax=30
        n_per_seg = 250             # window to do power spectra
        n_fft = n_per_seg + 50      # for the FFT I consider a padding of 0 at the edges, of 25 samples per side
        n_overlap = n_per_seg//2

        stime = 1 * sfreq
        etime = 1 * sfreq

        kargs = {"sfreq":sfreq, "fmin":fmin, "fmax":fmax,
                "n_per_seg":n_per_seg, "n_fft":n_fft,
                "n_overlap":n_overlap}

        #  return features_MI, features_Rest, labels_MI, labels_Rest, selected_regions
        Feat_extraction = extract_features_more_bands(data_2_MI, data_2_Rest, 
                                        sfreq, starttime=stime, endtime=etime,
                                        min_freq_frature=miff, max_freq_frature=maff, 
                                        nbest_regions=num_best_ROIs, kargs_welch=kargs)

        Feat_MI, Feat_Rest, labls_MI, labls_Rest, selected_regions = Feat_extraction
        Feat_MI.shape
        #####################################################################################################################

        training = train_models(np.concatenate((Feat_MI,Feat_Rest)),np.concatenate((labls_MI,labls_Rest)),
                                method=meth, 
                                seed=537)
        new_row = {
            "ROIs": [selected_regions],
            "NROIs":[len(selected_regions)],
            "DataType": [DataType],
            "freqs_band": [freqs_band],  
            "subject": [subject_index],
            "Classifier": [Classifier_name], #SVC, #RandomForest, #LDA
            "Performance": [training[0]],
            "Features": [Features_name],
            "Comments": [Comments],
            "Date": [today],
            "Personalized_Freq_Range": [PFR],
            
        }
        Rows=pd.concat((Rows,pd.DataFrame(new_row)))


if saveresults:
    saveresults_pickle(Rows, inputfile=filepath, outputfile=filepath, backup=True)



 Subject 0
Effective window size : 1.200 (s)
Effective window size : 1.200 (s)
###################################################################### 
 Exception occurred: [Errno 2] No such file or directory: '/Users/giovanni.messuti/Desktop/BCI_Project/Results/BUTTARE.pkl' 
 ######################################################################
Cannot Read a previous file, creating a new one at /Users/giovanni.messuti/Desktop/BCI_Project/Results/BUTTARE.pkl


In [ ]:
# Integrate EEG MEG
import mne
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from datetime import date
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA

today = date.today()
sfreq = 250


Rows = pd.DataFrame({
    "ROIs": [],
    "NROIs":[],
    "DataType": [],
    "freqs_band": [],  
    "subject": [],
    "Classifier": [],
    "Performance": [],
    "Features": [],
    "Comments": [],
    "Date": []
})

# keys of the performance dictionary
DataType="EEG concat MEG"
saveresults = True
filepath = "/Users/giovanni.messuti/Desktop/BCI_Project/Results/BCI_Performances.pkl"
inputfile=filepath; outputfile=filepath
miff = [4,8,12]
maff = [8,12,30]
freqs_band = tuple([(i,j) for i,j in zip(miff,maff)])
Classifier_name = "RandomForest" #SVC, #RandomForest, #LDA
meth = RandomForestClassifier
Features_name = "PowerSpectra - MeanMax band"
Comments= "No Interesting Regions" # "-"
PFR = False


for subject_index in range(20):
    data_2_MI_EEG = read_subject(data_folder, "EEG", "MI",subject_index, verbose=False)
    data_2_Rest_EEG =read_subject(data_folder, "EEG", "Baseline",subject_index, verbose=False)
    data_2_MI_MEG = read_subject(data_folder, "MEG", "MI",subject_index, verbose=False)
    data_2_Rest_MEG =read_subject(data_folder, "MEG", "Baseline",subject_index, verbose=False)
    
    # miff = freq_personalized[f"subject_{subject_index}"][0]
    # maff = freq_personalized[f"subject_{subject_index}"][-1]
    # freqs_band = (miff, maff)

    for num_best_ROIs in range(8,69,4):
        print(f"\n\n Subject {subject_index}")
        #####################################################################################################################
        fmin=3
        fmax=30
        n_per_seg = 250             # window to do power spectra
        n_fft = n_per_seg + 50      # for the FFT I consider a padding of 0 at the edges, of 25 samples per side
        n_overlap = n_per_seg//2

        stime = 1 * sfreq
        etime = 1 * sfreq

        kargs = {"sfreq":sfreq, "fmin":fmin, "fmax":fmax,
                "n_per_seg":n_per_seg, "n_fft":n_fft,
                "n_overlap":n_overlap}

        #  return features_MI, features_Rest, labels_MI, labels_Rest, selected_regions
        Feat_extraction_EEG = extract_features_more_bands(data_2_MI_EEG, data_2_Rest_EEG, 
                                        sfreq, starttime=stime, endtime=etime,
                                        min_freq_frature=miff, max_freq_frature=maff, 
                                        nbest_regions=num_best_ROIs, kargs_welch=kargs,
                                        important_regions=[],
                                        )

        Feat_MI_EEG, Feat_Rest_EEG, labls_MI_EEG, labls_Rest_EEG, selected_regions_EEG = Feat_extraction_EEG
    
        Feat_extraction_MEG = extract_features_more_bands(data_2_MI_MEG, data_2_Rest_MEG, 
                                        sfreq, starttime=stime, endtime=etime,
                                        min_freq_frature=miff, max_freq_frature=maff, 
                                        nbest_regions=num_best_ROIs, kargs_welch=kargs,
                                        important_regions=[],
                                        )

        Feat_MI_MEG, Feat_Rest_MEG, labls_MI_MEG, labls_Rest_MEG, selected_regions_MEG = Feat_extraction_MEG

        Features_MI = np.concatenate((Feat_MI_EEG,Feat_MI_MEG), axis=1)
        Features_Rest = np.concatenate((Feat_Rest_EEG,Feat_Rest_MEG), axis=1)
        Features = np.concatenate((Features_MI,Features_Rest))
        Labels = np.concatenate((labls_MI_EEG,labls_Rest_EEG))
        #print(Feat_MI_MEG.shape, Features_MI.shape, Features.shape, labls_MI_EEG.shape, Labels.shape)
        #####################################################################################################################

        training = train_models(Features,Labels,
                                method=meth, 
                                seed=537,
                                verbose=True)
        new_row = {
            "ROIs": [[selected_regions_EEG]+[selected_regions_MEG]],
            "NROIs":[len(selected_regions_EEG)],
            "DataType": [DataType],
            "freqs_band": [freqs_band],  
            "subject": [subject_index],
            "Classifier": [Classifier_name], #SVC, #RandomForest, #LDA
            "Performance": [training[0]],
            "Features": [Features_name],
            "Comments": [Comments],
            "Date": [today],
            "Personalized_Freq_Range": [PFR],
            
        }
        Rows=pd.concat((Rows,pd.DataFrame(new_row)))



if saveresults:
    saveresults_pickle(Rows, inputfile=filepath, outputfile=filepath, backup=True)



 Subject 0
Effective window size : 1.200 (s)
Effective window size : 1.200 (s)
Effective window size : 1.200 (s)
Effective window size : 1.200 (s)

----- Fold 1 -----
Fold 1 accuracy: 0.9688, Confusion matrix:
[[19  0]
 [ 1 12]]

----- Fold 2 -----
Fold 2 accuracy: 1.0000, Confusion matrix:
[[17  0]
 [ 0 14]]

----- Fold 3 -----
Fold 3 accuracy: 0.9677, Confusion matrix:
[[12  1]
 [ 0 18]]

----- Fold 4 -----
Fold 4 accuracy: 0.9355, Confusion matrix:
[[15  0]
 [ 2 14]]

----- Fold 5 -----
Fold 5 accuracy: 0.9677, Confusion matrix:
[[13  1]
 [ 0 17]]

Mean accuracy: 0.9679435483870968
Std accuracy: 0.020405775691741946


 Subject 0
Effective window size : 1.200 (s)
Effective window size : 1.200 (s)
Effective window size : 1.200 (s)
Effective window size : 1.200 (s)

----- Fold 1 -----
Fold 1 accuracy: 1.0000, Confusion matrix:
[[19  0]
 [ 0 13]]

----- Fold 2 -----
Fold 2 accuracy: 1.0000, Confusion matrix:
[[17  0]
 [ 0 14]]

----- Fold 3 -----
Fold 3 accuracy: 0.9677, Confusion mat

# Test

In [ ]:
import pandas as pd 

filepath = "/Users/giovanni.messuti/Desktop/BCI_Project/Results/BCI_Performances.pkl"
results=pd.read_pickle(filepath)
results


,ROIs,NROIs,DataType,freqs_band,subject,Classifier,Performance,Features,Comments,Date,Personalized_Freq_Range
0,"[4, 5, 32, 33, 44, 45, 48, 49, 23, 15, 14, 22,...",14.0,EEG,"((4, 8), (8, 13), (13, 25))",0.0,LDA,"[0.78125, 0.6774193548387096, 0.83870967741935...",PowerSpectra - MeanMax band,-,2025-11-20,False
1,"[4, 5, 32, 33, 44, 45, 48, 49, 23, 15, 14, 22,...",17.0,EEG,"((4, 8), (8, 13), (13, 25))",0.0,LDA,"[0.8125, 0.6774193548387096, 0.580645161290322...",PowerSpectra - MeanMax band,-,2025-11-20,False
2,"[4, 5, 32, 33, 44, 45, 48, 49, 23, 15, 14, 22,...",20.0,EEG,"((4, 8), (8, 13), (13, 25))",0.0,LDA,"[0.75, 0.5806451612903226, 0.6129032258064516,...",PowerSpectra - MeanMax band,-,2025-11-20,False
3,"[4, 5, 32, 33, 44, 45, 48, 49, 23, 15, 14, 22,...",23.0,EEG,"((4, 8), (8, 13), (13, 25))",0.0,LDA,"[0.8125, 0.6451612903225806, 0.612903225806451...",PowerSpectra - MeanMax band,-,2025-11-20,False
4,"[4, 5, 32, 33, 44, 45, 48, 49, 23, 15, 14, 22,...",26.0,EEG,"((4, 8), (8, 13), (13, 25))",0.0,LDA,"[0.75, 0.6774193548387096, 0.8387096774193549,...",PowerSpectra - MeanMax band,-,2025-11-20,False
...,...,...,...,...,...,...,...,...,...,...,...
3415,"[[4, 5, 32, 33, 44, 45, 48, 49, 14, 59, 63, 51...",56.0,EEG concat MEG,"((4, 8), (8, 13), (13, 25))",19.0,LDA,"[0.8421052631578947, 0.7368421052631579, 0.710...",PowerSpectra - MeanMax band,ROIs: EEG then MEG,2025-11-25,False
3416,"[[4, 5, 32, 33, 44, 45, 48, 49, 14, 59, 63, 51...",59.0,EEG concat MEG,"((4, 8), (8, 13), (13, 25))",19.0,LDA,"[0.7894736842105263, 0.7368421052631579, 0.736...",PowerSpectra - MeanMax band,ROIs: EEG then MEG,2025-11-25,False
3417,"[[4, 5, 32, 33, 44, 45, 48, 49, 14, 59, 63, 51...",62.0,EEG concat MEG,"((4, 8), (8, 13), (13, 25))",19.0,LDA,"[0.8157894736842105, 0.7368421052631579, 0.763...",PowerSpectra - MeanMax band,ROIs: EEG then MEG,2025-11-25,False
3418,"[[4, 5, 32, 33, 44, 45, 48, 49, 14, 59, 63, 51...",65.0,EEG concat MEG,"((4, 8), (8, 13), (13, 25))",19.0,LDA,"[0.7894736842105263, 0.8421052631578947, 0.736...",PowerSpectra - MeanMax band,ROIs: EEG then MEG,2025-11-25,False


In [4]:
OK_results = results[results["DataType"]!="EEG concat MEG"]

In [2]:
import mne
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from datetime import date
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA

today = date.today()
sfreq = 250


Rows = pd.DataFrame({
    "ROIs": [],
    "NROIs":[],
    "DataType": [],
    "freqs_band": [],  
    "subject": [],
    "Classifier": [],
    "Performance": [],
    "Features": [],
    "Comments": [],
    "Date": []
})

# keys of the performance dictionary
saveresults = True
#filepath = "/Users/giovanni.messuti/Desktop/BCI_Project/Results/BCI_Performances.pkl"
DataType="EEG"
miff = [4,8,12]
maff = [8,12,30]
freqs_band = tuple([(i,j) for i,j in zip(miff,maff)])
Classifier_name = "LDA" #SVC, #RandomForest, #LDA
meth = LDA
Features_name = "PowerSpectra - MeanMax band"
Comments= "Only mean" # "-"
PFR = False


for subject_index in range(20):
        for num_best_ROIs in range(14,69,3):
                print(f"\n\n Subject {subject_index}")
                data_2_MI = read_subject(data_folder, DataType, "MI",subject_index, verbose=False)
                data_2_Rest =read_subject(data_folder, DataType, "Baseline",subject_index, verbose=False)
                #####################################################################################################################
                fmin= 3
                fmax=30
                n_per_seg = 250             # window to do power spectra
                n_fft = n_per_seg + 50      # for the FFT I consider a padding of 0 at the edges, of 25 samples per side
                n_overlap = n_per_seg//2

                stime = 1 * sfreq
                etime = 1 * sfreq

                kargs = {"sfreq":sfreq, "fmin":fmin, "fmax":fmax,
                        "n_per_seg":n_per_seg, "n_fft":n_fft,
                        "n_overlap":n_overlap}

                #  return features_MI, features_Rest, labels_MI, labels_Rest, selected_regions
                Feat_extraction = extract_features_more_bands(data_2_MI, data_2_Rest, 
                                                sfreq, starttime=stime, endtime=etime,
                                                min_freq_frature=miff, max_freq_frature=maff, 
                                                nbest_regions=num_best_ROIs, kargs_welch=kargs,
                                                select_max=False)

                Feat_MI, Feat_Rest, labls_MI, labls_Rest, selected_regions = Feat_extraction
                Feat_MI.shape
                #####################################################################################################################

                training = train_models(np.concatenate((Feat_MI,Feat_Rest)),np.concatenate((labls_MI,labls_Rest)),
                                        method=meth, 
                                        seed=537)
                new_row = {
                    "ROIs": [selected_regions],
                    "NROIs":[len(selected_regions)],
                    "DataType": [DataType],
                    "freqs_band": [freqs_band],  
                    "subject": [subject_index],
                    "Classifier": [Classifier_name], #SVC, #RandomForest, #LDA
                    "Performance": [training[0]],
                    "Features": [Features_name],
                    "Comments": [Comments],
                    "Date": [today],
                    "Personalized_Freq_Range": [PFR],
                    
                }
                Rows=pd.concat((Rows,pd.DataFrame(new_row)))



if saveresults:
    saveresults_pickle(Rows, inputfile=None, outputfile="/Users/giovanni.messuti/Desktop/BCI_Project/Results/BCI_Performances_ony_mean.pkl", backup=True)



 Subject 0
Effective window size : 1.200 (s)
Effective window size : 1.200 (s)


 Subject 0
Effective window size : 1.200 (s)
Effective window size : 1.200 (s)


 Subject 0
Effective window size : 1.200 (s)
Effective window size : 1.200 (s)


 Subject 0
Effective window size : 1.200 (s)
Effective window size : 1.200 (s)


 Subject 0
Effective window size : 1.200 (s)
Effective window size : 1.200 (s)


 Subject 0
Effective window size : 1.200 (s)
Effective window size : 1.200 (s)


 Subject 0
Effective window size : 1.200 (s)
Effective window size : 1.200 (s)


 Subject 0
Effective window size : 1.200 (s)
Effective window size : 1.200 (s)


 Subject 0
Effective window size : 1.200 (s)
Effective window size : 1.200 (s)


 Subject 0
Effective window size : 1.200 (s)
Effective window size : 1.200 (s)


 Subject 0
Effective window size : 1.200 (s)
Effective window size : 1.200 (s)


 Subject 0
Effective window size : 1.200 (s)
Effective window size : 1.200 (s)


 Subject 0
Effective windo